In [0]:
from __future__ import annotations

import sys
from pathlib import Path

from pyspark.sql import DataFrame
from pyspark.sql import functions as F

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "notebooks" / "common").exists():
        repository_root = str(candidate)
        if repository_root not in sys.path:
            sys.path.insert(0, repository_root)
        break

from notebooks.common.metrics import (
    create_pipeline_metric_df,
    print_summary,
)
from notebooks.config.northstar.paths import PATHS

In [0]:
NOTEBOOK_VERSION = "1.0.0"
PIPELINE_NAME = "northstar_gold_eligibility_reconciliation"

BUSINESS_DATE = "2026-07-01"

# Silver
SILVER_EMPLOYEE_PATH = PATHS.silver_path("employees")
SILVER_DEPENDENT_PATH = PATHS.silver_path("dependents")
SILVER_ENROLLMENT_PATH = PATHS.silver_path("enrollments")
SILVER_ELIGIBILITY_PATH = (
    f"abfss://silver@{PATHS.storage_account}.dfs.core.windows.net/"
    "northstar/eligibility"
)

# Gold
GOLD_ROOT = (
    f"abfss://gold@{PATHS.storage_account}.dfs.core.windows.net/"
    "northstar"
)

GOLD_RECONCILIATION_PATH = (
    f"{GOLD_ROOT}/eligibility_reconciliation"
)

GOLD_RECONCILIATION_KPI_PATH = (
    f"{GOLD_ROOT}/eligibility_reconciliation_kpis"
)

METRICS_PATH = (
    f"{GOLD_ROOT}/data_quality_metrics"
)

print(SILVER_EMPLOYEE_PATH)
print(SILVER_DEPENDENT_PATH)
print(SILVER_ENROLLMENT_PATH)
print(SILVER_ELIGIBILITY_PATH)

In [0]:
employees_df = (
    spark.read
    .format("delta")
    .load(SILVER_EMPLOYEE_PATH)
)

dependents_df = (
    spark.read
    .format("delta")
    .load(SILVER_DEPENDENT_PATH)
)

enrollments_df = (
    spark.read
    .format("delta")
    .load(SILVER_ENROLLMENT_PATH)
)

eligibility_df = (
    spark.read
    .format("delta")
    .load(SILVER_ELIGIBILITY_PATH)
)

print(f"Employees:   {employees_df.count():,}")
print(f"Dependents:  {dependents_df.count():,}")
print(f"Enrollments: {enrollments_df.count():,}")
print(f"Eligibility: {eligibility_df.count():,}")

In [0]:
enrollment_keys_df = (
    enrollments_df
    .select(
        "employee_id",
        "dependent_id",
        "plan_id",
        "enrollment_status",
        "coverage_start_date",
        "coverage_end_date",
    )
    .withColumn(
        "person_id",
        F.coalesce(
            F.col("dependent_id"),
            F.col("employee_id"),
        ),
    )
    .withColumn(
        "person_type",
        F.when(
            F.col("dependent_id").isNotNull(),
            F.lit("Dependent"),
        ).otherwise(F.lit("Employee")),
    )
)

eligibility_keys_df = (
    eligibility_df
    .select(
        "employee_id",
        "dependent_id",
        "plan_id",
        "eligibility_status",
        "eligibility_start_date",
        "eligibility_end_date",
        "approval_timestamp",
    )
    .withColumn(
        "person_id",
        F.coalesce(
            F.col("dependent_id"),
            F.col("employee_id"),
        ),
    )
    .withColumn(
        "person_type",
        F.when(
            F.col("dependent_id").isNotNull(),
            F.lit("Dependent"),
        ).otherwise(F.lit("Employee")),
    )
)

print(f"Enrollment keys:  {enrollment_keys_df.count():,}")
print(f"Eligibility keys: {eligibility_keys_df.count():,}")

In [0]:
reconciliation_df = (
    eligibility_keys_df.alias("elig")
    .join(
        enrollment_keys_df.alias("enr"),
        on=[
            F.col("elig.employee_id") == F.col("enr.employee_id"),
            F.coalesce(
                F.col("elig.dependent_id"),
                F.lit(""),
            )
            == F.coalesce(
                F.col("enr.dependent_id"),
                F.lit(""),
            ),
            F.col("elig.plan_id") == F.col("enr.plan_id"),
        ],
        how="full_outer",
    )
    .select(
        F.coalesce(
            F.col("elig.employee_id"),
            F.col("enr.employee_id"),
        ).alias("employee_id"),

        F.coalesce(
            F.col("elig.dependent_id"),
            F.col("enr.dependent_id"),
        ).alias("dependent_id"),

        F.coalesce(
            F.col("elig.person_id"),
            F.col("enr.person_id"),
        ).alias("person_id"),

        F.coalesce(
            F.col("elig.person_type"),
            F.col("enr.person_type"),
        ).alias("person_type"),

        F.coalesce(
            F.col("elig.plan_id"),
            F.col("enr.plan_id"),
        ).alias("plan_id"),

        F.col("elig.eligibility_status"),
        F.col("enr.enrollment_status"),

        F.col("elig.eligibility_start_date"),
        F.col("elig.eligibility_end_date"),

        F.col("enr.coverage_start_date"),
        F.col("enr.coverage_end_date"),

        F.col("elig.approval_timestamp"),
    )
    .withColumn(
        "reconciliation_status",
        F.when(
            F.col("eligibility_status").isNotNull()
            & F.col("enrollment_status").isNotNull(),
            F.lit("MATCHED"),
        )
        .when(
            F.col("eligibility_status").isNotNull()
            & F.col("enrollment_status").isNull(),
            F.lit("ELIGIBLE_NOT_ENROLLED"),
        )
        .when(
            F.col("eligibility_status").isNull()
            & F.col("enrollment_status").isNotNull(),
            F.lit("ENROLLED_NOT_ELIGIBLE"),
        )
        .otherwise(F.lit("UNKNOWN")),
    )
    .withColumn(
        "business_date",
        F.to_date(F.lit(BUSINESS_DATE)),
    )
    .withColumn(
        "processing_timestamp",
        F.current_timestamp(),
    )
)

display(
    reconciliation_df.groupBy("reconciliation_status")
    .count()
    .orderBy("reconciliation_status")
)

In [0]:
matched_count = (
    reconciliation_df
    .filter(F.col("reconciliation_status") == "MATCHED")
    .count()
)

eligible_not_enrolled_count = (
    reconciliation_df
    .filter(
        F.col("reconciliation_status") == "ELIGIBLE_NOT_ENROLLED"
    )
    .count()
)

enrolled_not_eligible_count = (
    reconciliation_df
    .filter(
        F.col("reconciliation_status") == "ENROLLED_NOT_ELIGIBLE"
    )
    .count()
)

total_records = reconciliation_df.count()

executive_kpi_df = spark.createDataFrame(
    [
        (
            BUSINESS_DATE,
            total_records,
            matched_count,
            eligible_not_enrolled_count,
            enrolled_not_eligible_count,
            round(matched_count / total_records * 100, 2),
        )
    ],
    [
        "business_date",
        "total_records",
        "matched_records",
        "eligible_not_enrolled",
        "enrolled_not_eligible",
        "match_rate_pct",
    ],
)

display(executive_kpi_df)

In [0]:
(
    reconciliation_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_RECONCILIATION_PATH)
)

reconciliation_written_count = (
    spark.read
    .format("delta")
    .load(GOLD_RECONCILIATION_PATH)
    .count()
)

print(
    f"Detailed reconciliation rows written: "
    f"{reconciliation_written_count:,}"
)

In [0]:
(
    executive_kpi_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_RECONCILIATION_KPI_PATH)
)

kpi_written_count = (
    spark.read
    .format("delta")
    .load(GOLD_RECONCILIATION_KPI_PATH)
    .count()
)

if kpi_written_count != 1:
    raise RuntimeError(
        "Reconciliation KPI validation failed: "
        f"expected=1, actual={kpi_written_count:,}"
    )

print("Reconciliation KPI rows written: 1")

In [0]:
expected_reconciliation_count = (
    matched_count
    + eligible_not_enrolled_count
    + enrolled_not_eligible_count
)

if reconciliation_written_count != expected_reconciliation_count:
    raise RuntimeError(
        "Gold reconciliation validation failed: "
        f"expected={expected_reconciliation_count:,}, "
        f"actual={reconciliation_written_count:,}"
    )

print("\nEligibility reconciliation completed successfully.")
print(f"Notebook version:              {NOTEBOOK_VERSION}")
print(f"Detailed reconciliation rows:  {reconciliation_written_count:,}")
print(f"Matched records:               {matched_count:,}")
print(f"Eligible not enrolled:         {eligible_not_enrolled_count:,}")
print(f"Enrolled not eligible:         {enrolled_not_eligible_count:,}")
print(f"Match rate:                    {matched_count / reconciliation_written_count * 100:.2f}%")
print(f"Reconciliation path:           {GOLD_RECONCILIATION_PATH}")
print(f"KPI path:                      {GOLD_RECONCILIATION_KPI_PATH}")